# Cochleogram-ViT — Audio-level Augmentation (librosa backend)

Pre-computes a NEW cochleogram set with the librosa-mel backend (faster than
pycochleagram, lets us augment cheaply). The original
`data/processed/cochleograms/` (pycochleagram) is **untouched**.

Two new dirs are written:
- `data/processed/cochleograms_librosa/`         (clean, librosa) — used for val + baseline train
- `data/processed/cochleograms_librosa_aug/`     (3 augmented variants/sample) — train only

**Audio augmentations** (applied to the raw waveform before mel extraction):
- random gain (±3 dB), prob 0.8
- additive Gaussian noise (SNR 25-40 dB), prob 0.5
- time stretch (rate 0.9-1.1), prob 0.5
- pitch shift (±1 semitone), prob 0.2

Two training passes for a clean ablation, each 10-fold (paper metric):
1. **librosa-baseline**: no aug — establishes the librosa-backend baseline.
2. **librosa-aug**: train picks one of {clean, 3 augmented} per sample per epoch; val stays clean.

Both use the sweep's best config: weighted loss, SOFTEN_POWER=0.25, no KAN.

NOTE: the **pre-compute cells take ~60-80 min** the first time (single-threaded
with progress). They're resume-safe (skip files that already exist).


In [1]:
# --- Imports + paths ---
import os, json, time, copy, gc, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import torch
import torch.optim as optim
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
import librosa
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.model_selection import GroupKFold, StratifiedGroupKFold
from sklearn.utils.class_weight import compute_class_weight

from cochleogram_vit.models.vit import CochleogramViT


# ---- paths (originals untouched) ----
ORIG_DIR        = '../data/processed/cochleograms'              # pycochleagram (untouched)
CLEAN_DIR       = '../data/processed/cochleograms_librosa'      # NEW: librosa clean
AUG_DIR         = '../data/processed/cochleograms_librosa_aug'  # NEW: librosa + audio aug
METADATA_PATH   = '../data/processed/metadata.csv'
RESULTS_PATH    = '../results/audioaug.json'
PREDS_PATH      = '../results/audioaug_preds.npz'

os.makedirs(CLEAN_DIR, exist_ok=True)
os.makedirs(AUG_DIR, exist_ok=True)
os.makedirs(os.path.dirname(RESULTS_PATH), exist_ok=True)

# ---- audio + cochleogram params (match original pipeline as closely as possible) ----
SR            = 22050
CLIP_SEC      = 5.0
CLIP_LEN      = int(SR * CLIP_SEC)
N_MELS        = 128
FMIN, FMAX    = 50, 8000
OUTPUT_SIZE   = 128
N_AUG         = 3      # augmented variants per sample

# ---- training params (match sweep's best config: loss-p025) ----
SOFTEN_POWER  = 0.25
BATCH_SIZE    = 16
EPOCHS        = 30
LEARNING_RATE = 1e-4

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'device: {device}')
md = pd.read_csv(METADATA_PATH)
md['patient_id'] = md['npy_path'].apply(lambda x: os.path.basename(x).split('_')[0])
print(f'metadata rows: {len(md)}')


device: cuda
metadata rows: 6898


In [2]:
# --- librosa mel-cochleogram + audio augmentation ---

def load_clip(wav_path, start_s, end_s):
    """Load wav, mono, resample to SR, crop [start,end], pad/truncate to CLIP_LEN samples."""
    waveform, orig_sr = torchaudio.load(wav_path)
    if waveform.shape[0] > 1:
        waveform = waveform.mean(dim=0, keepdim=True)
    if orig_sr != SR:
        waveform = torchaudio.transforms.Resample(orig_sr, SR)(waveform)
    y = waveform.squeeze(0).numpy()
    s = int(start_s * SR); e = int(end_s * SR)
    y = y[s:e]
    if len(y) < CLIP_LEN:
        y = np.pad(y, (0, CLIP_LEN - len(y)))
    else:
        y = y[:CLIP_LEN]
    return y.astype(np.float32)


def mel_cochleogram(y):
    """librosa mel-spectrogram -> log -> normalize [0,1] -> resize to (128,128)."""
    mel = librosa.feature.melspectrogram(y=y, sr=SR, n_mels=N_MELS, fmin=FMIN, fmax=FMAX, power=2.0)
    log_mel = librosa.power_to_db(mel, ref=np.max)
    lm = (log_mel - log_mel.min()) / (log_mel.max() - log_mel.min() + 1e-8)
    # resize to square
    t = torch.from_numpy(lm.astype(np.float32))[None, None]
    t = F.interpolate(t, size=(OUTPUT_SIZE, OUTPUT_SIZE), mode='bilinear', align_corners=False)
    return t.squeeze().numpy()


def apply_audio_aug(y, rng):
    """Sequence of random audio transforms (probabilistic)."""
    # gain
    if rng.random() < 0.8:
        y = y * (10 ** (rng.uniform(-3, 3) / 20))
    # additive noise
    if rng.random() < 0.5:
        snr_db = rng.uniform(25, 40)
        sig_p = float(np.mean(y**2)) + 1e-10
        noise_p = sig_p / (10 ** (snr_db / 10))
        y = y + rng.normal(0, np.sqrt(noise_p), size=y.shape).astype(np.float32)
    # time stretch (changes length; mel will resize back to fixed shape)
    if rng.random() < 0.5:
        rate = float(rng.uniform(0.9, 1.1))
        y = librosa.effects.time_stretch(y, rate=rate)
    # pitch shift (slow; low probability)
    if rng.random() < 0.2:
        n_steps = float(rng.uniform(-1, 1))
        y = librosa.effects.pitch_shift(y, sr=SR, n_steps=n_steps)
    # re-pad/truncate after potential length change
    if len(y) < CLIP_LEN:
        y = np.pad(y, (0, CLIP_LEN - len(y)))
    else:
        y = y[:CLIP_LEN]
    return y.astype(np.float32)


In [3]:
# --- Pre-compute CLEAN librosa cochleograms (wav-cached, resume-safe) ---
# Each wav has ~7 cycles; load+resample each wav ONCE rather than per cycle.
needed = {}
for _, row in md.iterrows():
    name = os.path.splitext(os.path.basename(row['npy_path']))[0]
    out  = os.path.join(CLEAN_DIR, f'{name}.npy')
    if not os.path.exists(out):
        wav = os.path.join('..', row['wav_path'])
        needed.setdefault(wav, []).append((out, float(row['start']), float(row['end'])))
n_files = sum(len(v) for v in needed.values())
print(f'wavs to load: {len(needed)}   clean cochleograms to generate: {n_files}/{len(md)}')

if needed:
    t0 = time.time()
    pbar = tqdm(total=n_files, desc='clean')
    for wav, tasks in needed.items():
        waveform, orig_sr = torchaudio.load(wav)
        if waveform.shape[0] > 1: waveform = waveform.mean(dim=0, keepdim=True)
        if orig_sr != SR: waveform = torchaudio.transforms.Resample(orig_sr, SR)(waveform)
        full_y = waveform.squeeze(0).numpy().astype(np.float32)
        for out_path, start, end in tasks:
            s, e = int(start * SR), int(end * SR)
            y = full_y[s:e]
            if len(y) < CLIP_LEN: y = np.pad(y, (0, CLIP_LEN - len(y)))
            else:                 y = y[:CLIP_LEN]
            np.save(out_path, mel_cochleogram(y).astype(np.float32))
            pbar.update(1)
    pbar.close()
    print(f'clean done in {(time.time()-t0)/60:.1f} min')
else:
    print('clean already up to date')


wavs to load: 920   clean cochleograms to generate: 6898/6898


clean: 100%|██████████| 6898/6898 [02:01<00:00, 56.88it/s] 

clean done in 2.0 min


In [4]:
# --- Pre-compute AUGMENTED librosa cochleograms (wav-cached, resume-safe) ---
needed = {}
for i, row in md.iterrows():
    name = os.path.splitext(os.path.basename(row['npy_path']))[0]
    wav  = os.path.join('..', row['wav_path'])
    for k in range(N_AUG):
        out = os.path.join(AUG_DIR, f'{name}_aug{k}.npy')
        if not os.path.exists(out):
            needed.setdefault(wav, []).append((out, float(row['start']), float(row['end']), hash((i, k)) & 0x7fffffff))
n_files = sum(len(v) for v in needed.values())
print(f'wavs to load: {len(needed)}   aug cochleograms to generate: {n_files}/{len(md)*N_AUG}')

if needed:
    t0 = time.time()
    pbar = tqdm(total=n_files, desc='aug')
    for wav, tasks in needed.items():
        waveform, orig_sr = torchaudio.load(wav)
        if waveform.shape[0] > 1: waveform = waveform.mean(dim=0, keepdim=True)
        if orig_sr != SR: waveform = torchaudio.transforms.Resample(orig_sr, SR)(waveform)
        full_y = waveform.squeeze(0).numpy().astype(np.float32)
        for out_path, start, end, seed in tasks:
            s, e = int(start * SR), int(end * SR)
            y = full_y[s:e]
            if len(y) < CLIP_LEN: y = np.pad(y, (0, CLIP_LEN - len(y)))
            else:                 y = y[:CLIP_LEN]
            rng = np.random.default_rng(seed)
            y = apply_audio_aug(y, rng)
            np.save(out_path, mel_cochleogram(y).astype(np.float32))
            pbar.update(1)
    pbar.close()
    print(f'aug done in {(time.time()-t0)/60:.1f} min')
else:
    print('aug already up to date')


wavs to load: 920   aug cochleograms to generate: 20694/20694


aug: 100%|██████████| 20694/20694 [09:11<00:00, 37.53it/s]

aug done in 9.2 min


In [5]:
# --- Dataset that switches between clean and (clean U augmented) per sample ---
class LibrosaCochDataset(Dataset):
    """
    augment=False -> always returns the clean librosa cochleogram (for val + baseline).
    augment=True  -> randomly returns clean OR one of N_AUG variants per __getitem__
                     (so each epoch sees a different mix; clean is included as one option).
    """
    def __init__(self, metadata, clean_dir, aug_dir, augment=False, n_aug=N_AUG):
        self.metadata = metadata
        self.clean_dir = clean_dir
        self.aug_dir = aug_dir
        self.augment = augment
        self.n_aug = n_aug
        self._viridis = plt.get_cmap('viridis')

    def __len__(self):
        return len(self.metadata)

    def __getitem__(self, idx):
        row = self.metadata.iloc[idx]
        name = os.path.splitext(os.path.basename(row['npy_path']))[0]
        if self.augment:
            # uniformly pick clean (0) or one of n_aug variants (1..n_aug)
            k = np.random.randint(0, self.n_aug + 1)
            if k == 0:
                path = os.path.join(self.clean_dir, f'{name}.npy')
            else:
                path = os.path.join(self.aug_dir, f'{name}_aug{k-1}.npy')
        else:
            path = os.path.join(self.clean_dir, f'{name}.npy')
        coch = np.load(path)
        rgb = self._viridis(coch)[:, :, :3].transpose(2, 0, 1)
        return torch.from_numpy(np.ascontiguousarray(rgb)).float(), int(row['label'])


# CV setup (shared across both passes)
gkf = StratifiedGroupKFold(n_splits=10, shuffle=True, random_state=42)  # was GroupKFold
FOLDS = list(gkf.split(md, md['label'].values, groups=md['patient_id'].values))
print(f'StratifiedGroupKFold: {len(FOLDS)} folds')


def softened_weights(softpow):
    raw = compute_class_weight('balanced', classes=np.array([0,1,2,3]), y=md['label'].values)
    w = raw ** softpow
    return w / w.sum() * len(w)

def lr_lambda(epoch):
    warmup = 4
    if epoch < warmup:
        return (epoch + 1) / warmup
    denom = EPOCHS - warmup
    return 0.5 * (1 + np.cos(np.pi * (epoch - warmup) / denom)) if denom > 0 else 0.0

def paper_metrics(preds, labels):
    p = np.asarray(preds); y = np.asarray(labels)
    TP   = int(np.sum((y != 0) & (p == y)))
    FN   = int(np.sum((y != 0) & (p == 0)))
    FN_w = int(np.sum((y != 0) & (p != 0) & (p != y)))
    TN   = int(np.sum((y == 0) & (p == 0)))
    FP   = int(np.sum((y == 0) & (p != 0)))
    TP_b = TP + FN_w
    se = TP_b / (TP_b + FN + 1e-8)
    sp = TN / (TN + FP + 1e-8)
    return {'TP': TP, 'FN': FN, 'FN_wrong': FN_w, 'TN': TN, 'FP': FP,
            'se': float(se), 'sp': float(sp), 'score': float((se + sp) / 2)}

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    preds, labels, probs = [], [], []
    for x, y in loader:
        x = x.to(device)
        logits = model(x)
        probs.append(torch.softmax(logits, dim=1).cpu().numpy())
        preds.extend(logits.argmax(dim=1).cpu().numpy().tolist())
        labels.extend(y.numpy().tolist())
    return preds, labels, np.concatenate(probs, axis=0)


StratifiedGroupKFold: 10 folds


In [6]:
# --- Shared 10-fold training runner ---
def run_pass(run_id, train_augment):
    print(f'\n{"="*70}\n[{run_id}]  train_augment={train_augment}\n{"="*70}')
    t_start = time.time()
    cw = softened_weights(SOFTEN_POWER)
    cw_t = torch.tensor(cw, dtype=torch.float).to(device)
    print(f'class weights (^{SOFTEN_POWER}): {np.round(cw, 3).tolist()}')

    train_ds = LibrosaCochDataset(md, CLEAN_DIR, AUG_DIR, augment=train_augment)
    val_ds   = LibrosaCochDataset(md, CLEAN_DIR, AUG_DIR, augment=False)

    fold_rows, pooled_preds, pooled_labels = [], [], []
    per_fold_probs, per_fold_labels, per_fold_val_idx = {}, {}, {}

    for fold, (train_idx, val_idx) in enumerate(FOLDS):
        torch.manual_seed(42 + fold); np.random.seed(42 + fold)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(42 + fold)

        train_loader = DataLoader(Subset(train_ds, train_idx), batch_size=BATCH_SIZE, shuffle=True)
        val_loader   = DataLoader(Subset(val_ds,   val_idx),   batch_size=BATCH_SIZE, shuffle=False)

        model = CochleogramViT(image_size=128, patch_size=16, num_classes=4,
                               dim=512, depth=6, heads=8, mlp_dim=1024, channels=3,
                               dropout=0.3, emb_dropout=0.2).to(device)
        opt   = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
        sched = optim.lr_scheduler.LambdaLR(opt, lr_lambda)
        criterion = nn.CrossEntropyLoss(weight=cw_t)

        best_score = -1.0; best_state = None; best_epoch = 0
        for epoch in range(EPOCHS):
            model.train()
            for x, y in train_loader:
                x, y = x.to(device), y.to(device)
                opt.zero_grad()
                loss = criterion(model(x), y)
                loss.backward()
                opt.step()
            preds, labels, _ = evaluate(model, val_loader)
            m = paper_metrics(preds, labels)
            if m['score'] > best_score:
                best_score = m['score']; best_state = copy.deepcopy(model.state_dict()); best_epoch = epoch + 1
            sched.step()

        model.load_state_dict(best_state)
        preds, labels, probs = evaluate(model, val_loader)
        m = paper_metrics(preds, labels)
        fold_rows.append({'fold': fold + 1, 'best_epoch': best_epoch, **m})
        pooled_preds.extend(preds); pooled_labels.extend(labels)
        per_fold_probs[f'fold{fold+1}_probs']  = probs.astype(np.float32)
        per_fold_labels[f'fold{fold+1}_labels'] = np.asarray(labels, dtype=np.int64)
        per_fold_val_idx[f'fold{fold+1}_val_idx'] = np.asarray(val_idx, dtype=np.int64)
        print(f'  fold {fold+1:>2}: best_ep={best_epoch:>2}  Se={m["se"]*100:5.2f}  Sp={m["sp"]*100:5.2f}  Score={m["score"]*100:5.2f}')

        del model, opt, sched, train_loader, val_loader
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    pooled = paper_metrics(pooled_preds, pooled_labels)
    se_mean = float(np.mean([r['se']    for r in fold_rows]))
    sp_mean = float(np.mean([r['sp']    for r in fold_rows]))
    sc_mean = float(np.mean([r['score'] for r in fold_rows]))
    sc_std  = float(np.std ([r['score'] for r in fold_rows]))
    elapsed = time.time() - t_start
    print(f'\n  PER-FOLD MEAN: Se={se_mean*100:.2f}  Sp={sp_mean*100:.2f}  Score={sc_mean*100:.2f}  std={sc_std*100:.2f}')
    print(f'  POOLED:        Se={pooled["se"]*100:.2f}  Sp={pooled["sp"]*100:.2f}  Score={pooled["score"]*100:.2f}')
    print(f'  elapsed: {elapsed/60:.1f} min')

    return {
        'id': run_id,
        'config': {'soften_power': SOFTEN_POWER, 'train_augment': train_augment,
                   'backend': 'librosa_mel', 'epochs': EPOCHS, 'batch_size': BATCH_SIZE, 'lr': LEARNING_RATE,
                   'n_aug': N_AUG},
        'elapsed_sec': elapsed,
        'folds': fold_rows,
        'per_fold_mean': {'se': se_mean, 'sp': sp_mean, 'score': sc_mean, 'score_std': sc_std},
        'pooled_aggregate': pooled,
        'preds': {**per_fold_probs, **per_fold_labels, **per_fold_val_idx},
    }


In [7]:
# --- Two passes (resume-safe at the run level) ---
results = []
if os.path.exists(RESULTS_PATH):
    try:
        results = json.load(open(RESULTS_PATH))
        print(f'loaded {len(results)} prior runs')
    except Exception as e:
        print(f'could not load prior results ({e}); starting fresh')
        results = []
done = {r.get('id') for r in results}

RUNS = [
    ('librosa-baseline', False),  # baseline on librosa backend, no aug
    ('librosa-aug',      True),   # librosa backend + audio augmentation
]

all_preds = {}
for run_id, augment in RUNS:
    if run_id in done:
        print(f'[{run_id}] already done, skipping')
        continue
    r = run_pass(run_id, augment)
    # split preds out of JSON (npz is more compact for arrays)
    preds = r.pop('preds')
    all_preds[run_id] = preds
    results.append(r)
    with open(RESULTS_PATH, 'w') as f:
        json.dump(results, f, indent=2, default=str)
    # incremental save of preds (one npz per run to keep things simple)
    np.savez(PREDS_PATH.replace('.npz', f'_{run_id}.npz'), **preds)
    print(f'  -> {RESULTS_PATH}  +  preds npz')

print('\nALL PASSES COMPLETE')



[librosa-baseline]  train_augment=False
class weights (^0.25): [0.763, 0.902, 1.086, 1.249]
[CochleogramViT] Parameters — total: 13,040,644  trainable: 13,040,644
  fold  1: best_ep=13  Se=46.43  Sp=75.21  Score=60.82
[CochleogramViT] Parameters — total: 13,040,644  trainable: 13,040,644
  fold  2: best_ep=20  Se=62.01  Sp=53.85  Score=57.93
[CochleogramViT] Parameters — total: 13,040,644  trainable: 13,040,644
  fold  3: best_ep=23  Se=56.91  Sp=56.71  Score=56.81
[CochleogramViT] Parameters — total: 13,040,644  trainable: 13,040,644
  fold  4: best_ep= 1  Se=24.92  Sp=95.88  Score=60.40
[CochleogramViT] Parameters — total: 13,040,644  trainable: 13,040,644
  fold  5: best_ep=12  Se=56.16  Sp=66.30  Score=61.23
[CochleogramViT] Parameters — total: 13,040,644  trainable: 13,040,644
  fold  6: best_ep=15  Se=39.80  Sp=82.09  Score=60.94
[CochleogramViT] Parameters — total: 13,040,644  trainable: 13,040,644
  fold  7: best_ep= 5  Se=67.73  Sp=51.91  Score=59.82
[CochleogramViT] Paramete

In [8]:
# --- Summary table (this notebook + sweep for reference) ---
print(f"{'config':<28} {'per-fold Sc±std':<18} {'pooled':<8} {'elapsed':<8}")
print('-' * 68)
for r in results:
    if 'error' in r: continue
    m, p = r['per_fold_mean'], r['pooled_aggregate']
    print(f"{r['id']:<28} {m['score']*100:5.2f}±{m['score_std']*100:4.2f}        {p['score']*100:.2f}     {r['elapsed_sec']/60:5.1f}m")

sweep_path = '../results/sweep_overnight.json'
if os.path.exists(sweep_path):
    print('\n--- reference (sweep, pycochleagram backend, NOT directly comparable) ---')
    for r in json.load(open(sweep_path)):
        if 'error' in r: continue
        m, p = r['per_fold_mean'], r['pooled_aggregate']
        print(f"{r['id']:<28} {m['score']*100:5.2f}±{m['score_std']*100:4.2f}        {p['score']*100:.2f}")


config                       per-fold Sc±std    pooled   elapsed 
--------------------------------------------------------------------
librosa-baseline             60.55±2.76        60.42      46.5m
librosa-aug                  60.39±3.69        60.48      46.6m

--- reference (sweep, pycochleagram backend, NOT directly comparable) ---
loss-p05                     64.86±5.70        63.21
loss-p025                    65.25±5.49        64.67
sampler-p05                  64.27±4.82        64.31
kan-last-p05                 64.43±5.96        64.33
kan-first-p05                64.75±5.20        63.54
